In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# from typing import Optional

# from pylabrobot.resources.height_volume_functions import (
#   compute_height_from_volume_rectangle,
#   compute_volume_from_height_rectangle,
# )
# from pylabrobot.resources.plate import Lid, Plate
# from pylabrobot.resources.utils import create_ordered_items_2d
# from pylabrobot.resources.well import (
#   CrossSectionType,
#   Well,
#   WellBottomType,
# )

# this treats the tuberack like a plate. Quick hack, but not appropriate for pull request and long term.
# works 20250719

# def opentrons_24_tuberack_generic_1point5ml_snapcap_short(name: str) -> Plate:
#   """
#   OpenTrons 24 well rack with the shorter stand
#   3D print available here: https://www.thingiverse.com/thing:3405002
#   Spec sheet (json):
#   https://raw.githubusercontent.com/Opentrons/opentrons/edge/shared-data/labware/definitions/2/opentrons_24_tuberack_nest_1.5ml_screwcap/1.json
#   """
#   INNER_WELL_WIDTH = 9.2  # measured  
#   INNER_WELL_LENGTH = 9.2  # measured

#   well_kwargs = {
#     "size_x": INNER_WELL_WIDTH,  # measured
#     "size_y": INNER_WELL_LENGTH,  # measured
#     "size_z": 37.40,  # measured
#     "bottom_type": WellBottomType.V,
#     "cross_section_type": CrossSectionType.RECTANGLE,
#     "compute_height_from_volume": lambda liquid_volume: compute_height_from_volume_rectangle(
#       liquid_volume,
#       INNER_WELL_LENGTH,
#       INNER_WELL_WIDTH,
#     ),
#     "compute_volume_from_height": lambda liquid_height: compute_volume_from_height_rectangle(
#       liquid_height,
#       INNER_WELL_LENGTH,
#       INNER_WELL_WIDTH,
#     ),
#     # "material_z_thickness": 1,
#     "material_z_thickness": 0.80, # measured
#   }

#   return Plate(
#     name=name,
#     size_x=127.75,  # from spec
#     size_y=85.50,  # from spec
#     size_z=48.5,  # measured (this is the shorter platform
#     model=opentrons_24_tuberack_generic_1point5ml_snapcap_short.__name__,
#     ordered_items=create_ordered_items_2d(
#       Well,
#       num_items_x=6,
#       num_items_y=4,
#       dx=12.5,  # measured
#       dy=16.5,  # measured
#       dz=18,  # measured
#       item_dx=19.89, # from spec
#       item_dy=19.28, # from spec
#       **well_kwargs,
#     ),
#   )

In [3]:
# # import in this notebook so I can iterate and play with the settings.

# # from pylabrobot.resources.tuberack import TubeRack   # or your local import path
# from pylabrobot.resources.tube_rack import TubeRack
# from pylabrobot.resources.tube import Tube
# # from pylabrobot.resources import CrossSectionType, WellBottomType
# from pylabrobot.resources.utils import create_ordered_items_2d
# )

# def opentrons_24_tuberack_generic_1point5ml_snapcap_short(name: str) -> TubeRack:
#     INNER_WELL_WIDTH  = 9.2
#     INNER_WELL_LENGTH = 9.2
#     WELL_DEPTH        = 37.40
#     TUBE_MAX_VOL      = 1750  # µL  (generic 1.75 mL snap-cap)

#     tube_kwargs = {
#         "size_x": INNER_WELL_WIDTH,
#         "size_y": INNER_WELL_LENGTH,
#         "size_z": WELL_DEPTH,        
#         "max_volume": TUBE_MAX_VOL,
#         "material_z_thickness": 0.80
#         }

#     return TubeRack(
#         name=name,
#         size_x=127.75,
#         size_y=85.50,
#         size_z=48.5,
#         model=opentrons_24_tuberack_generic_1point5ml_snapcap_short.__name__,
#         ordered_items=create_ordered_items_2d(
#             Tube,
#             num_items_x=6,
#             num_items_y=4,
#             dx=12.5,
#             dy=15.8,
#             dz=12,
#             item_dx=19.89,
#             item_dy=19.28,
#             **tube_kwargs,
#         ),
#     )

In [4]:

# ── imports ──────────────────────────────────────────────
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    MFX_CAR_L5_base,
    TIP_CAR_480_A00
)
from pylabrobot.resources.hamilton.mfx_modules import (
    MFX_DWP_module_188042
)
from pylabrobot.resources import (
    TIP_50ul_w_filter,
                 HTF             # 50 µL filter tip rack
)
import asyncio

# ── build LH + deck ──────────────────────────────────────
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())

# ── initialise hardware (Autoload) ───────────────────────
await lh.setup(skip_autoload=True)

# ── tip carrier with 50 µL tips ───────────────────────────
tip_car = TIP_CAR_480_A00("tip_car")
tip_car[0] = HTF(name="tips_00")
tip_car[1] = TIP_50ul_w_filter(name="tips_01")
lh.deck.assign_child_resource(tip_car, rails=25)


# ── module → carrier → deck ──────────────────────────────
dwp_mod   = MFX_DWP_module_188042("dwp_mod_1")
flex_car  = MFX_CAR_L5_base("flex_car_1", modules={0: dwp_mod})
lh.deck.assign_child_resource(flex_car, rails=13)

In [5]:
# ── pick up a single 50 µL tip on channel 3 ─────────────
tiprack = lh.deck.get_resource("tips_01")
longrack = lh.deck.get_resource("tips_00")

# await lh.pick_up_tips(longrack["B4"], use_channels=[5])
await lh.pick_up_tips(tiprack["E8"], use_channels=[5])

In [6]:
from pylabrobot.resources.opentrons.tube_racks import opentrons_24_tuberack_generic_1point5ml_snapcap_short

In [7]:

from pylabrobot.resources.tube_adapter import TubeRackAdapter

rack = opentrons_24_tuberack_generic_1point5ml_snapcap_short("rack1")
offset_x = (127.76 - rack._size_x) / 2
offset_y = (85.48  - rack._size_y) / 2

adapter = TubeRackAdapter(
    name="rack7_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=rack._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=offset_x,
    dy=offset_y,
    dz=0,
    adapter_hole_size_x=rack._size_x,
    adapter_hole_size_y=rack._size_y,
    adapter_hole_size_z=rack._size_z
)

adapter.assign_child_resource(rack)
dwp_mod.assign_child_resource(adapter)


await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(adapter["A1"], vols=[0], use_channels=[5])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(adapter["D1"], vols=[0], use_channels=[5])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(adapter["A6"], vols=[0], use_channels=[5])
# await lh.prepare_for_manual_channel_operation(3)
await lh.aspirate(adapter["D6"], vols=[0], use_channels=[5])
await lh.prepare_for_manual_channel_operation(3)



In [8]:
# await lh.drop_tips(tiprack["D1"], use_channels=[5])
await lh.discard_tips()
await lh.stop()